# 실습 2-1. 사출 데이터 정제

가상 공장의 사출기 3대(IMM-01·02·03)가 PLC 로 올린 **샷 단위 성형 조건** 5,000여 행을 정제함.
이 데이터에는 현장에서 흔히 생기는 오류 다섯 종이 섞여 있음.

| 오류 | 가상 공장에서 본 원인 |
|---|---|
| 결측 | 금형온도 센서 단선 |
| 이상치 | 통신 오류 값(0, 999.9), 자릿수 밀림 |
| 단위 혼재 | 한 설비만 압력을 bar 로 전송 |
| 시각 오차 | 한 설비의 PLC 시계가 어긋남 |
| 중복 | 게이트웨이 재전송 |

- **기본 트랙**: 셀을 위에서부터 차례로 실행(`Shift` + `Enter`)하고, 각 단계의 결과를 워크북 규칙표에 옮겨 적음.
- **심화 트랙**: `# 심화` 표시가 있는 값을 바꿔 보고 결과가 어떻게 달라지는지 확인함.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "common").exists():      # 저장소 최상위 폴더 찾기
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from common.plot_style import use_korean_font
use_korean_font()

DATA = ROOT / "data" / "session2"
OUT = DATA / "work"                        # 내 산출물이 저장되는 곳
OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 160)

## 1. 수집 검증: 무엇이 들어왔는지 확인

파일은 두 개임. 성형 조건은 사출기 PLC 가, 검사 결과는 외관 검사기가 따로 올림.

In [ ]:
shots = pd.read_csv(DATA / "injection_shots_raw.csv", parse_dates=["timestamp"])
insp = pd.read_csv(DATA / "inspection_results.csv", parse_dates=["inspect_time"])
print("성형 조건:", shots.shape, "  검사 결과:", insp.shape)
shots.head()

In [ ]:
insp.head()

In [ ]:
COND = ["mold_temp_c", "melt_temp_c", "inj_pressure_mpa", "hold_pressure_mpa",
        "hold_time_s", "cooling_time_s", "cycle_time_s"]
KO = {"mold_temp_c": "금형온도(℃)", "melt_temp_c": "수지온도(℃)", "inj_pressure_mpa": "사출압(MPa)",
      "hold_pressure_mpa": "보압(MPa)", "hold_time_s": "보압 시간(s)", "cooling_time_s": "냉각 시간(s)",
      "cycle_time_s": "사이클타임(s)"}
shots[COND].describe().T.round(1)

> **읽어 보기**: 최솟값·최댓값이 상식과 맞는지 봄. 금형온도 최댓값 999.9, 수지온도 최솟값 0, 보압 최댓값 500 이상은 정상 값이 아님.

규칙을 정할 때마다 아래 `rules` 목록에 한 줄씩 기록함. 이것이 산출물인 **정제 규칙표**가 됨.

In [ ]:
rules = []          # 정제 규칙표
def log_rule(error, how_found, rule, n):
    rules.append({"오류 유형": error, "찾은 방법": how_found, "처리 규칙": rule, "해당 행 수": int(n)})
df = shots.copy()

## 2. 중복: 같은 행이 두 번 들어옴

게이트웨이가 응답을 못 받으면 같은 데이터를 다시 보냄. 설비·시각·샷 번호가 모두 같은 행은 하나만 남김.

In [ ]:
dup = df.duplicated(subset=["machine_id", "timestamp", "shot_no"], keep="first")
print("중복 행:", dup.sum())
df[df.duplicated(subset=["machine_id", "timestamp", "shot_no"], keep=False)].head(4)

In [ ]:
df = df[~dup].reset_index(drop=True)
log_rule("중복", "설비·시각·샷 번호가 같은 행", "첫 행만 남김", dup.sum())
print("남은 행:", len(df))

## 3. 단위 혼재: 한 설비만 값이 10배

설비별로 압력 분포를 나란히 그려 봄.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, col in zip(axes, ["inj_pressure_mpa", "hold_pressure_mpa"]):
    ok = df[df[col] < 990]                      # 통신 오류 값은 잠시 제외하고 봄
    ax.boxplot([ok.loc[ok.machine_id == m, col].dropna() for m in ["IMM-01", "IMM-02", "IMM-03"]],
               tick_labels=["IMM-01", "IMM-02", "IMM-03"])
    ax.set_title(KO[col] + "  (원본)")
plt.tight_layout(); plt.show()
df.groupby("machine_id")[["inj_pressure_mpa", "hold_pressure_mpa"]].median()

> IMM-03 만 약 10배임. 이 설비는 압력을 **bar** 로 올리고 있음(1 MPa = 10 bar). MPa 로 맞춤.

In [ ]:
is_bar = df["machine_id"] == "IMM-03"
df.loc[is_bar, ["inj_pressure_mpa", "hold_pressure_mpa"]] /= 10.0
log_rule("단위 혼재", "설비별 압력 분포 비교 (IMM-03 만 10배)", "IMM-03 의 사출압·보압을 10 으로 나눠 MPa 로 통일", is_bar.sum())
df.groupby("machine_id")[["inj_pressure_mpa", "hold_pressure_mpa"]].median().round(1)

## 4. 이상치: 있을 수 없는 값

항목마다 **현장에서 나올 수 있는 범위**를 정하고, 벗어난 값은 "값 없음"으로 바꿈.
범위는 설비 사양과 공정 담당자의 경험으로 정함.

In [ ]:
# 심화: 범위를 좁히거나 넓혀 보고 걸리는 건수가 어떻게 달라지는지 확인
VALID = {
    "mold_temp_c": (20, 120),
    "melt_temp_c": (150, 300),
    "inj_pressure_mpa": (30, 200),
    "hold_pressure_mpa": (10, 150),
    "hold_time_s": (1, 30),
    "cooling_time_s": (5, 90),
    "cycle_time_s": (30, 200),
}
n_out = 0
for col, (lo, hi) in VALID.items():
    bad = (df[col] < lo) | (df[col] > hi)
    if bad.sum():
        print(f"{KO[col]:14s} 범위 {lo}~{hi} 밖: {bad.sum():3d}건   예) {sorted(df.loc[bad, col].unique())[:4]}")
    df.loc[bad, col] = np.nan
    n_out += bad.sum()
log_rule("이상치", "항목별 허용 범위 밖 (0, 999.9, 10배 값)", "값을 비우고 결측으로 처리", n_out)

## 5. 결측: 값이 비어 있음

어느 항목이, 어느 설비에서, 언제 비었는지부터 봄.

In [ ]:
print(df[COND].isna().sum())
miss = df[df["mold_temp_c"].isna()]
miss.groupby(["machine_id", miss["timestamp"].dt.date]).size().rename("금형온도 결측 수")

In [ ]:
# 금형온도 결측이 시간상 어떻게 몰려 있는지
fig, ax = plt.subplots(figsize=(10, 2.6))
for i, m in enumerate(["IMM-01", "IMM-02", "IMM-03"]):
    d = df[df.machine_id == m]
    ax.scatter(d["timestamp"], np.full(len(d), i), s=2, color="#bbbbbb")
    d = d[d["mold_temp_c"].isna()]
    ax.scatter(d["timestamp"], np.full(len(d), i), s=14, color="crimson")
ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["IMM-01", "IMM-02", "IMM-03"])
ax.set_title("금형온도 결측 위치 (빨간 점)"); plt.tight_layout(); plt.show()

> 낱개로 빠진 값과 **수십 샷이 연달아 빠진 구간**(센서 단선)이 섞여 있음. 둘은 다르게 처리함.
>
> - 짧게 빠진 값: 같은 설비의 앞뒤 값으로 이어 채움 (조건은 샷 사이에 천천히 변하므로)
> - 길게 빠진 구간: 채우면 지어낸 값이 되므로 그 행을 쓰지 않음

In [ ]:
MAX_FILL = 3        # 심화: 연속 몇 개까지 채울지 (0 이면 채우지 않음)
before = df[COND].isna().sum().sum()
df = df.sort_values(["machine_id", "timestamp"]).reset_index(drop=True)
df[COND] = df.groupby("machine_id")[COND].transform(
    lambda s: s.interpolate(limit=MAX_FILL, limit_area="inside") if MAX_FILL > 0 else s)
# 긴 구간은 앞에서부터 MAX_FILL 개만 채워지므로, 구간 길이가 MAX_FILL 을 넘으면 채운 값도 되돌림
raw_na = shots.drop_duplicates(["machine_id", "timestamp", "shot_no"]).sort_values(["machine_id", "timestamp"]).reset_index(drop=True)["mold_temp_c"].isna()
run_id = (raw_na != raw_na.shift()).cumsum()
long_gap = raw_na & (raw_na.groupby(run_id).transform("size") > MAX_FILL)
df.loc[long_gap.values, "mold_temp_c"] = np.nan
filled = before - df[COND].isna().sum().sum()
drop = df[COND].isna().any(axis=1)
print(f"채운 값 {filled}개,  쓰지 않는 행 {drop.sum()}개")
df["usable"] = ~drop           # 행은 지우지 않고 표시만 해 둠 (시각 보정에 필요)
log_rule("결측", "항목별 빈 값 수, 설비·시각별 분포", f"연속 {MAX_FILL}개 이하는 앞뒤 값으로 보간, 그보다 긴 구간은 행 제외", drop.sum() + filled)

## 6. 시각 오차: 설비 시계가 어긋남

성형 조건과 검사 결과에는 공통 번호가 없어서 **설비 번호와 시각**으로 묶어야 함.
사출 후 검사기까지 걸리는 시간은 설비 위치에 따라 다름(취출 10초 + 컨베이어 이동).

| 설비 | 검사기까지 거리 | 걸리는 시간 |
|---|---|---|
| IMM-01 | 11.5 m | 154초 |
| IMM-02 | 7.5 m | 104초 |
| IMM-03 | 3.5 m | 54초 |

시계가 어긋난 설비가 있으면 엉뚱한 부품끼리 묶임.
설비·날짜별로 "그날 첫 검사 시각 − 첫 샷 시각"을 구해 봄. 정상이라면 위 표의 값 근처여야 함.

In [ ]:
first_shot = df.groupby(["machine_id", df["timestamp"].dt.date])["timestamp"].min()
first_insp = insp.groupby(["machine_id", insp["inspect_time"].dt.date])["inspect_time"].min()
gap = (first_insp - first_shot).dt.total_seconds().unstack(0)
gap.index.name = "날짜"
gap.round(0)

> IMM-02 는 11월 23일부터 값이 약 −328초임. 104 − (−328) = **432초(7분 12초)** 만큼 PLC 시계가 빨라진 것임.
> 그날 PLC 를 재기동하면서 시각 동기화가 풀린 경우에 해당함. 서버 시각 기준으로 되돌림.

In [ ]:
NORMAL_DELAY = pd.Series({"IMM-01": 154, "IMM-02": 104, "IMM-03": 54})   # 사출 → 검사 소요 시간(초)
offset = (NORMAL_DELAY - gap).round(0)              # 설비·날짜별 시계 오차
bad = offset[offset.abs() > 30].stack().dropna()
print("보정이 필요한 설비·날짜\n", bad, sep="")
n_fix = 0
for (day, machine), sec in bad.items():
    sel = (df["machine_id"] == machine) & (df["timestamp"].dt.date == day)
    df.loc[sel, "timestamp"] -= pd.Timedelta(seconds=float(sec))
    n_fix += sel.sum()
log_rule("시각 오차", "설비·날짜별 (첫 검사 시각 − 첫 샷 시각) 이 정상 소요 시간에서 벗어남",
         "IMM-02 의 11-23 이후 시각에서 432초를 뺌", n_fix)

## 7. 라벨 결합: 조건과 검사 결과를 묶음

샷 시각에 설비별 소요 시간을 더한 시각과 가장 가까운 검사 기록을 같은 설비 안에서 찾음. 20초 넘게 차이 나면 묶지 않음.

In [ ]:
TOLERANCE = 20      # 심화: 허용 오차(초)를 바꿔 결합률이 어떻게 달라지는지 확인
left = df.assign(expect_time=df["timestamp"] + pd.to_timedelta(df["machine_id"].map(NORMAL_DELAY), unit="s")).sort_values("expect_time")
right = insp.sort_values("inspect_time")
merged = pd.merge_asof(left, right, left_on="expect_time", right_on="inspect_time", by="machine_id",
                       direction="nearest", tolerance=pd.Timedelta(seconds=TOLERANCE))
print(f"결합된 행: {merged['result'].notna().sum()} / {len(merged)}  ({merged['result'].notna().mean()*100:.1f}%)")

## 8. 특성 생성과 저장

- `is_defect`: 불량이면 1
- `cooling_ratio`: 사이클타임 중 냉각 시간의 비율 (생산량을 맞추려고 냉각을 줄였는지 보는 값)

In [ ]:
clean = merged[merged["usable"] & merged["result"].notna()].copy()
clean["is_defect"] = (clean["result"] == "NG").astype(int)
clean["cooling_ratio"] = (clean["cooling_time_s"] / clean["cycle_time_s"]).round(3)
clean = clean.sort_values(["timestamp", "machine_id"])[
    ["timestamp", "machine_id", "lot_id", "shot_no", *COND, "cooling_ratio", "ambient_temp_c", "defect_type", "is_defect"]]
clean.to_csv(OUT / "injection_clean.csv", index=False)
rule_table = pd.DataFrame(rules)
rule_table.to_csv(OUT / "cleaning_rules.csv", index=False, encoding="utf-8-sig")
print(f"원본 {len(shots)}행 → 정제본 {len(clean)}행,  불량률 {clean['is_defect'].mean()*100:.1f}%")
rule_table

## 9. 확인: 정제 전과 후

같은 그림을 정제 전·후로 나란히 놓고 봄.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 5.5))
for j, col in enumerate(["mold_temp_c", "inj_pressure_mpa", "hold_pressure_mpa"]):
    axes[0, j].hist(shots[col].dropna(), bins=60, color="#c0392b"); axes[0, j].set_title("정제 전  " + KO[col])
    axes[1, j].hist(clean[col], bins=60, color="#2471a3"); axes[1, j].set_title("정제 후  " + KO[col])
plt.tight_layout(); plt.show()

### 워크북에 옮겨 적기

1. 위의 **정제 규칙표** 다섯 줄 (오류 유형 · 찾은 방법 · 처리 규칙 · 해당 행 수)
2. 원본 행 수와 정제본 행 수
3. 우리 회사 데이터라면 다섯 오류 중 어떤 것이 가장 자주 생길지 한 문장

저장된 파일: `data/session2/work/injection_clean.csv`, `data/session2/work/cleaning_rules.csv`